# 📊 Executive Sales Data Analytics & Business Intelligence

## Portfolio Project Overview
This notebook presents an end-to-end exploratory and diagnostic data analysis of multi-year retail transactions.
We investigate revenue patterns, profit drivers, regional variations, discount sensitivity, and customer segments to derive actionable business strategies.

### Core Analytical Objectives:
1. **Revenue & Profit Dynamics**: Evaluate growth trends and seasonal revenue spikes.
2. **Category & Product Economics**: Identify high-margin leaders vs. margin-eroding 'profit bleeders'.
3. **Discount Sensitivity**: Quantify how promotional discounting impacts net profitability.
4. **Geographic Distribution**: Compare performance across market regions and states.
5. **Strategic Recommendations**: Provide evidence-based operational decisions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set modern visual theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
pd.set_option("display.max_columns", None)

# Add src to module search path
sys.path.append(os.path.abspath("../src"))
from data_cleaning import clean_sales_data
from analysis import calculate_executive_kpis, analyze_monthly_trends, analyze_category_performance, analyze_discount_impact

## 1. Data Ingestion & Quality Audit
We load the raw multi-year transaction log to inspect its structural schema, missing fields, and potential duplicates.

In [ ]:
raw_csv_path = "../data/raw/sales_data.csv"
df_raw = pd.read_csv(raw_csv_path)
print(f"Raw records: {len(df_raw):,} rows | {df_raw.shape[1]} columns")
print(f"Duplicate rows detected: {df_raw.duplicated().sum()}")
print(f"Missing values per column:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}")
df_raw.head()

## 2. Data Cleaning & Feature Engineering
We apply our production ETL pipeline from `src/data_cleaning.py` to:
- Remove duplicate records.
- Parse datetime columns (`Order_Date`, `Ship_Date`).
- Compute `Shipping_Days`, `Order_Year`, `Order_Month`, `Profit_Margin %`, and `Discount_Tier`.

In [ ]:
df = clean_sales_data(raw_csv_path="../data/raw/sales_data.csv", processed_csv_path="../data/processed/sales_data_cleaned.csv")
df[["Order_ID", "Order_Date", "Category", "Sales", "Profit", "Profit_Margin", "Discount_Tier", "Shipping_Days"]].head()

## 3. Executive KPI Dashboard
Summary of company-wide financial and operational health.

In [ ]:
kpis = calculate_executive_kpis(df)
pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])

## 4. Time Series Trend Analysis
Tracking revenue and profitability across months and quarters.

In [ ]:
monthly = df.groupby("Year_Month").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()

plt.figure(figsize=(12, 5))
plt.plot(monthly["Year_Month"], monthly["Sales"], marker="o", color="#1f77b4", label="Sales ($)")
plt.plot(monthly["Year_Month"], monthly["Profit"], marker="s", color="#2ca02c", linestyle="--", label="Profit ($)")
plt.xticks(rotation=45, ha="right")
plt.title("Monthly Sales & Profit Performance", fontweight="bold")
plt.ylabel("Amount ($ USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Category & Sub-Category Diagnostics
Examining which products drive profitability vs. which cause profit leaks.

In [ ]:
cat_summary = df.groupby("Category").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()
cat_summary["Margin_%"] = (cat_summary["Profit"] / cat_summary["Sales"] * 100).round(2)
cat_summary

## 6. Discount Sensitivity & Margin Erosion
Testing the hypothesis: *Does heavier discounting drive volume at the expense of bottom-line profit?*

In [ ]:
disc_analysis = analyze_discount_impact(df)
print(disc_analysis)

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Discount_Tier", y="Profit_Margin", palette="vlag")
plt.axhline(0, color="red", linestyle="--", label="Break-Even Line (0% Margin)")
plt.title("Impact of Discount Tiers on Profit Margin %", fontweight="bold")
plt.ylabel("Profit Margin %")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Strategic Business Takeaways

1. **Enforce a 20% Discount Ceiling**: Transactions discounted above 20% consistently result in negative profit margins.
2. **Address Table Category Losses**: Tables operate with very high production costs and require restructuring or price floor enforcement.
3. **Capitalize on Q4 Seasonality**: Revenue surges by over 30% in Q4; inventory stocking and marketing promotions should concentrate between October and December.
4. **Expand Technology & Copiers**: Copiers and Technology hardware deliver the highest absolute margins across all regional markets.